# Trade Dependence, Economic Stability, and Democracy
## A Panel Data Analysis (1980–2024)

### Research Questions
1. How has trade dependence evolved over time?
2. Does trade dependence affect economic stability? 
3. Is a higher democratic indicator associated with lower macroeconomic volatility?

---

### Motivation

Globalization has increased dramatically over the past 40 years. 
However, the relationship between trade integration and macroeconomic stability remains debated.

Major arguments:
- Integrating into global supply chains fosters economic stability by reducing reliance on domestic factors and diversifying risk.
- Integrating economies into global supply chains increases their vulnerability to external shocks, as it tethers domestic stability to the unpredictable fluctuations of international markets.

---

##### This notebook investigates the empirical relationship between Trade dependence, GDP volatility and Inflation volatility, and further studies if there is an actual correlation between Democratic institutions and overall economic stability.


<br> <br>

### Data Sources

This analysis uses annual country-level data from:

- World Bank World Development Indicators:
  - GDP Growth (% annual)
  - Inflation (Consumer Prices, Annual)
  - Imports (% of GDP)
  - Exports (% of GDP)
  - Merchandise Trade (% of GDP)

- EIU Democracy Index:
  - Democracy Index (0–10 scale)

#### Time Period
1980–2024

#### Sample
- Countries with valid ISO-3 codes and available data.
- Countries with missing data due to certain political instabilities have not been dropped if their data is available in the post 2000 period. 
- Countries without valid data on the year 2000 are dropped assuming unavailability of data due to several factors.


<br> <br>

##### Importing relevant python libraries for our analysis

In [31]:
# Importing relevant libraries
import pandas as pd
import pycountry

##### Creating certain functions to assist data loading, data cleaning, dataframe melting, and the panel dataframe creation process 

In [56]:
#Function to load dataset into a pandas dataframe and clean metadata in the process
def data_loader (data_path, n):
    """
    Takes the path of the raw data csv file and loads it into a pandas dataframe, clearing metadata if required
    Takes two arguments - The data path for the csv file and the number of rows comprising metadata
    """
    dataframe = pd.read_csv(data_path, skiprows=n)
    return dataframe




# Function to clean World Bank data
def wb_data_cleaner(raw_data):
    """
    Takes the world bank raw data and processes it
    Takes only the raw data as argument
    """
    
    #dropping the columns not relevant for our analysis
    raw_data = raw_data.drop(
        columns=["Indicator Name", "Indicator Code", "Unnamed: 69"]) 
    # removing aggregates
    iso_codes = [country.alpha_3 for country in pycountry.countries]
    clean_data = raw_data[raw_data["Country Code"].isin(iso_codes)]
    # deleting unwanted columns, ie 1960 to 80
    years_to_remove = [str(year) for year in range(1960, 1980)]
    clean_data = clean_data.drop(columns=years_to_remove)
    # dropping countries whose data is missing in 2000 (mention in methodology)
    clean_data = clean_data.dropna(subset=["2000"])
    # reseting index
    clean_data = clean_data.reset_index(drop=True)
    return clean_data



# Function to clean EIU Democracy data
def eiu_data_cleaner(raw_data):
    '''
    Takes the EIU data and processes it in a relevant manner. 
    Takes only the raw data as argument.
    '''
    # dropping columns not relevant for our analysis
    raw_data = raw_data[["REF_AREA", "TIME_PERIOD","OBS_VALUE", "REF_AREA_LABEL"]]
    #renaming columns for our analysis
    raw_data = raw_data.rename(columns={'REF_AREA': 'Country Code', 'TIME_PERIOD': 'Year', 'OBS_VALUE': 'Democracy Index', 'REF_AREA_LABEL': 'Country Name'})
    raw_data["Year"] = raw_data["Year"].astype(int)
    #rearranging columns
    raw_data = raw_data[['Country Name', 'Country Code', 'Year', 'Democracy Index']]

    iso_codes = [country.alpha_3 for country in pycountry.countries]
    clean_data = raw_data[raw_data["Country Code"].isin(iso_codes)]
    clean_data = clean_data.reset_index(drop = True)
    
    return clean_data



# Function to convert world bank's wide format data into a long format one
def data_metler(wide_data, data_subject):
    '''
    Takes the processed dataframe in wide table format and converts it into long format
    Takes two arguments, the wide data and the specific value contained in the data
    '''
    long_data = wide_data.melt(
        id_vars=["Country Name", "Country Code"],
        var_name="Year",
        value_name=data_subject
    )
    long_data["Year"] = long_data["Year"].astype(int)
    return long_data



#Function to merge datasets and create a panel
def panel_creator(dataset1, dataset2, dataset3, dataset4, dataset5, dataset6):
    '''
    Takes the datasets (six for the purpose of this analysis) and merges them into a panel dataframe
    Takes the datasets to be merged as arguments
    '''
    merge_point = ["Country Name", "Country Code", "Year"]
    panel_df = dataset1.merge(dataset2, on = merge_point, how = "outer")
    panel_df = panel_df.merge(dataset3, on=merge_point, how="outer")
    panel_df = panel_df.merge(dataset4, on= merge_point, how="outer")
    panel_df = panel_df.merge(dataset5, on=merge_point, how="outer")
    #Casting the year as int for a smooth merge
    panel_df["Year"] = panel_df["Year"].astype(int) 
    panel_df = panel_df.merge(dataset6, on = merge_point, how="outer")
    #Sorting the data
    panel_df = panel_df.sort_values(["Country Code", "Year"])
    return panel_df
###explore reduce, lambda functions from the functools library

##### Loading the data from sources mentioned above

In [ ]:

#Assigning variables to data paths
gdp_data_path = "./data/raw/gdp_annual.csv"
inflation_data_path = "./data/raw/inflation_cp_a.csv"
imports_data_path = "./data/raw/imports_gs.csv"
exports_data_path = "./data/raw/exports_gs.csv"
merchandise_trade_data_path = "./data/raw/merchandise_trade_gdp.csv"
democracy_index_data_path = "./data/raw/EIU_DI.csv"


#Loading into pandas dataframes, devoid of metadata
gdp_data = data_loader(gdp_data_path, 4)
inflation_data = data_loader(inflation_data_path, 4)
imports_data = data_loader(imports_data_path, 4)
exports_data = data_loader(exports_data_path, 4)
merchandise_trade_data = data_loader(merchandise_trade_data_path, 4)
democracy_index_data = data_loader(democracy_index_data_path, 0)
democracy_index_data


##### Cleaning and processing the data using the created functions

In [51]:
#Cleaning the data as required using specific functions for WB and EIU data
gdp_data = wb_data_cleaner(gdp_data)
inflation_data = wb_data_cleaner(inflation_data)
imports_data = wb_data_cleaner(imports_data)
exports_data = wb_data_cleaner(exports_data)
merchandise_trade_data = wb_data_cleaner(merchandise_trade_data)
democracy_index_data = eiu_data_cleaner(democracy_index_data)


#Melting the WB dataframes into long form
gdp_data = data_metler(gdp_data, "GDP Growth")
inflation_data = data_metler(inflation_data, "Inflation annual")
imports_data = data_metler(imports_data, "Imports per GDP")
exports_data = data_metler(exports_data, "Exports per GDP")
merchandise_trade_data = data_metler(merchandise_trade_data, "Merchadise Trade per GDP")
#The EIU data is already in long format

##### Merging the various dataframes into a single sorted - panel dataset

In [ ]:
panel_dataframe = panel_creator(gdp_data, inflation_data, imports_data, exports_data, merchandise_trade_data, democracy_index_data)
panel_dataframe.head()

,Country Name,Country Code,Year,GDP Growth,Inflation annual,Imports per GDP,Exports per GDP,Merchadise Trade per GDP,Democracy Index
360,Aruba,ABW,1980,NaN,NaN,NaN,NaN,NaN,NaN
361,Aruba,ABW,1981,NaN,NaN,NaN,NaN,NaN,NaN
362,Aruba,ABW,1982,NaN,NaN,NaN,NaN,NaN,NaN
363,Aruba,ABW,1983,NaN,NaN,NaN,NaN,NaN,NaN
364,Aruba,ABW,1984,NaN,NaN,NaN,NaN,NaN,NaN


<br> <br> <br>